# Soccer Player Action Description Language (SPADL)

## Import Libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [ ]:
import warnings

import pandas as pd
from socceraction import spadl
from socceraction.data.statsbomb import StatsBombLoader
from tqdm import tqdm

from config import paths, tournaments

In [25]:
warnings.filterwarnings(
    "ignore",
    message="Inferred xy_fidelity_version=2. If this is incorrect, please specify the correct version using the xy_fidelity_version argument",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message="A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    category=pd.errors.PerformanceWarning,
)

## Load Match Data

In [4]:
# StatsBomb match ID for UEFA Euro 2024 Final
match_ids = tournaments.get_all_match_ids(tournaments.EURO_2024)
EURO_2024_FINAL_MATCH_ID = match_ids[0]

In [14]:
SBL = StatsBombLoader(getter="local", root=str(paths.STATSBOMB_DIR))
competitions_df = SBL.competitions()
teams_df = SBL.teams(game_id=EURO_2024_FINAL_MATCH_ID)
players_df = SBL.players(game_id=EURO_2024_FINAL_MATCH_ID)
events_df = SBL.events(game_id=EURO_2024_FINAL_MATCH_ID)

## Convert to SPADL

In [6]:
actions_df = spadl.statsbomb.convert_to_actions(
    events_df,
    home_team_id=teams_df["team_id"].iloc[0],
)

In [7]:
actions_df = spadl.add_names(actions_df).merge(teams_df).merge(players_df)

In [8]:
actions_df

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,...,result_name,bodypart_name,team_name,player_name,nickname,jersey_number,is_starter,starting_position_id,starting_position_name,minutes_played
0,3943043,152820f0-6ca9-4df3-943b-a67d568ff472,1,0.340,768,99174.0,52.54375,33.9575,82.81875,32.9375,...,success,foot_right,England,Kobbie Mainoo,None,26,True,9,Right Defensive Midfield,71
1,3943043,9c107df3-a3c8-4ad5-bc35-00214087a105,1,2.870,768,3468.0,82.81875,32.9375,79.93125,26.8175,...,success,foot,England,Jordan Pickford,None,1,True,1,Goalkeeper,96
2,3943043,237201b8-aef8-4823-b282-e82875795c07,1,4.742,768,3468.0,79.93125,26.8175,0.04375,57.5025,...,fail,foot_left,England,Jordan Pickford,None,1,True,1,Goalkeeper,96
3,3943043,238f44cb-0f18-4217-85b5-8cc6345278fe,1,34.440,772,11748.0,5.99375,34.3825,7.91875,19.4225,...,success,foot_left,Spain,Unai Simón Mendibil,Unai Simón,23,True,1,Goalkeeper,96
4,3943043,987a3f1e-4dc4-4833-8896-5513e35c3fc5,1,35.658,772,22128.0,7.91875,19.4225,7.74375,19.4225,...,success,foot,Spain,Robin Aime Robert Le Normand,Robin Le Normand,3,True,3,Right Center Back,84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1878,3943043,5d909b16-f11b-4d1c-8657-7eb5421ed102,2,2860.833,768,3468.0,54.73125,30.5575,12.46875,50.1075,...,fail,foot_left,England,Jordan Pickford,None,1,True,1,Goalkeeper,96
1879,3943043,4a610ec1-edf3-4066-bd69-cb2a28f2981c,2,2863.751,772,4353.0,12.46875,50.1075,10.89375,65.9175,...,success,head,Spain,Aymeric Laporte,None,14,True,5,Left Center Back,96
1880,3943043,c11721ac-1ddd-4a7d-8ee9-6ba005b6380f,2,2865.814,768,22084.0,10.89375,65.9175,12.73125,66.0025,...,success,foot,England,Bukayo Saka,None,7,True,17,Right Wing,96
1881,3943043,8eab51fc-0835-4fdb-8fe2-9c38fc38c46b,2,2867.318,768,22084.0,12.73125,66.0025,12.73125,66.0025,...,fail,foot,England,Bukayo Saka,None,7,True,17,Right Wing,96


## Load and convert ALL_TOURNAMENTS data

In [ ]:
all_tournaments_competition_name_list = [tournament["competition_name"] for tournament in tournaments.ALL_TOURNAMENTS]
all_tournaments_season_name_list = [tournament["season_name"] for tournament in tournaments.ALL_TOURNAMENTS]

selected_competitions_df = competitions_df[
    (competitions_df["competition_name"].isin(all_tournaments_competition_name_list))
    & (competitions_df["season_name"].isin(all_tournaments_season_name_list))
]

In [18]:
selected_competitions_df

,season_id,competition_id,competition_name,country_name,competition_gender,season_name
1,27,9,1. Bundesliga,Germany,male,2015/2016
21,282,223,Copa America,South America,male,2024
29,106,43,FIFA World Cup,International,male,2022
30,3,43,FIFA World Cup,International,male,2018
43,27,11,La Liga,Spain,male,2015/2016
60,27,7,Ligue 1,France,male,2015/2016
64,27,2,Premier League,England,male,2015/2016
66,27,12,Serie A,Italy,male,2015/2016
68,282,55,UEFA Euro,Europe,male,2024
69,43,55,UEFA Euro,Europe,male,2020


In [9]:
all_games_df = pd.concat(
    [
        SBL.games(
            int(tournament["competition_id"]),
            int(tournament["season_id"]),
        )
        for tournament in tournaments.ALL_TOURNAMENTS
    ]
)

In [10]:
all_games_df

,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee
0,3754058,27,2,Regular Season,20,2016-01-02 16:00:00,22,28,0,0,King Power Stadium,Andre Marriner
1,3754245,27,2,Regular Season,9,2015-10-17 16:00:00,27,41,1,0,The Hawthorns,Martin Atkinson
2,3754136,27,2,Regular Season,17,2015-12-19 18:30:00,37,59,1,1,St. James'' Park,Martin Atkinson
3,3754037,27,2,Regular Season,36,2016-04-30 16:00:00,29,28,2,1,Goodison Park,Neil Swarbrick
4,3754039,27,2,Regular Season,26,2016-02-13 16:00:00,31,23,1,2,Selhurst Park,Robert Madley
...,...,...,...,...,...,...,...,...,...,...,...,...
27,3939974,282,223,Group Stage,1,2024-06-24 01:00:00,1839,3564,2,0,AT&T Stadium,Maurizio Mariani
28,3939972,282,223,Group Stage,1,2024-06-23 01:00:00,3565,3563,1,2,Levi''s Stadium,Wilmar Alexander Roldán Pérez
29,3939971,282,223,Group Stage,1,2024-06-23 04:00:00,794,2305,1,0,NRG Stadium,Ismail Elfath
30,3939970,282,223,Group Stage,1,2024-06-22 03:00:00,784,3562,0,0,AT&T Stadium,Wilton Pereira Sampaio


In [ ]:
all_teams_list = []
all_players_list = []
game_actions_dict = {}

for game in tqdm(
    list(all_games_df.itertuples()),
    desc="Processing All Tournaments",
    ncols=150,
):
    all_teams_list.append(SBL.teams(game.game_id))  # type: ignore
    all_players_list.append(SBL.players(game.game_id))  # type: ignore
    game_events_df = SBL.events(game.game_id)  # type: ignore

    game_actions_dict[game.game_id] = spadl.statsbomb.convert_to_actions(
        game_events_df,
        home_team_id=game.home_team_id,  # type: ignore
        xy_fidelity_version=1,
        shot_fidelity_version=1,
    )

all_teams_df = pd.concat(all_teams_list).drop_duplicates(subset="team_id")
all_players_df = pd.concat(all_players_list)

Processing All Tournaments: 100%|█████████████████████████████████████████████████████████████████████████████████| 2085/2085 [33:07<00:00,  1.05it/s]


## Save SPADL data to HDF5

In [ ]:
with pd.HDFStore(paths.DATA_DIR / "socceraction" / "spadl.h5") as spadl_store:
    spadl_store["competitions"] = selected_competitions_df
    spadl_store["games"] = all_games_df
    spadl_store["teams"] = all_teams_df
    spadl_store["players"] = all_players_df[["player_id", "player_name", "nickname"]].drop_duplicates(
        subset="player_id"
    )
    spadl_store["player_games"] = all_players_df[
        [
            "player_id",
            "game_id",
            "team_id",
            "is_starter",
            "starting_position_id",
            "starting_position_name",
            "minutes_played",
        ]
    ]

    for game_id, game_actions in game_actions_dict.items():
        spadl_store[f"actions/game_{game_id}"] = game_actions